# Participant-dependent baseline and labeled calibration

This notebook simulates calibration for a new participant without touching the test set.

For every target participant:

1. Eyes-open baseline supplies spatial covariance alignment.
2. Eyes-closed baseline supplies an individual mu-frequency peak.
3. Acquisition trials supply optional feature centering and labeled final-layer adaptation.
4. Only later online trials are evaluated.

Calibration choices and prediction-blending weights are selected using the 55 training participants. The 12 validation participants are used only for the final development-stage report.

## 1. Imports and fixed experiment settings

Baseline recordings receive the same CAR, ICA/EOG, and residual EOG-regression cleaning as the task data. Conventional event markers are not required, so baseline files such as A16 and A30 can be processed as continuous recordings.

In [1]:
from pathlib import Path
import copy
import random
import time
import warnings

import mne
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy import linalg, signal
from sklearn.decomposition import FastICA
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

mne.set_log_level('ERROR')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate data/processed/Signals.')

DATA_ROOT = find_project_data()
SIGNALS_ROOT = DATA_ROOT / 'processed' / 'Signals'
PHYSIO_ROOT = DATA_ROOT / 'processed' / 'physiological_artifact_cleaning'
INDEX_PATH = PHYSIO_ROOT / 'development_index_physio_clean.csv'
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'participant_calibration'
BASELINE_CACHE_ROOT = OUTPUT_ROOT / 'baseline_by_participant'
PERSONAL_TASK_ROOT = OUTPUT_ROOT / 'individual_mu_covariances_by_file'
for directory in (OUTPUT_ROOT, BASELINE_CACHE_ROOT, PERSONAL_TASK_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

EEG_CHANNELS = [
    'Fz', 'FCz', 'Cz', 'CPz', 'Pz',
    'C1', 'C3', 'C5', 'C2', 'C4', 'C6',
    'F4', 'FC2', 'FC4', 'FC6', 'CP2', 'CP4', 'CP6', 'P4',
    'F3', 'FC1', 'FC3', 'FC5', 'CP1', 'CP3', 'CP5', 'P3',
]
EOG_CHANNELS = ['EOG1', 'EOG2', 'EOG3']
EMG_CHANNELS = ['EMGg', 'EMGd']
FIXED_BANDS = {'mu': (8.0, 13.0), 'beta': (13.0, 30.0)}
STIMULUS_SECONDS = (0.5, 5.0)
BASELINE_WINDOW_SECONDS = 2.0
ICA_COMPONENTS = 26
ICA_FIT_SAMPLES = 5_000
ICA_CORRELATION_THRESHOLD = 0.30
MAX_ICA_COMPONENTS = 3
N_CSP_FILTERS_PER_CLASS = 3
SHRINKAGE = 0.10
BATCH_SIZE = 128
GENERAL_EPOCHS = 15
ADAPTATION_EPOCHS = 20
ADAPTATION_LR = 1e-4

print('Output:', OUTPUT_ROOT)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/Users/aninanni/Documents/cosmos/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 807, in st

Output: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/participant_calibration


## 2. Load the cleaned development index and audit chronology

All participants must have acquisition trials and later online trials. Acquisition labels may be used for adaptation; online labels are reserved for evaluation.

In [2]:
development = pd.read_csv(INDEX_PATH)
assert set(development['split']) == {'train', 'validation'}
assert 'test' not in set(development['split'])
assert development['sample_id'].is_unique

participant_sets = {
    split: set(rows['participant'])
    for split, rows in development.groupby('split')
}
assert len(participant_sets['train']) == 55
assert len(participant_sets['validation']) == 12
assert participant_sets['train'].isdisjoint(participant_sets['validation'])

coverage = (
    development.groupby(['split', 'participant', 'phase']).size()
    .unstack(fill_value=0)
    .reset_index()
)
assert (coverage['acquisition'] > 0).all()
assert (coverage['online'] > 0).all()
display(coverage.groupby('split')[['acquisition', 'online']].describe())
print('Test participants loaded: 0')

phase      acquisition                                                       \
                 count       mean        std   min    25%   50%   75%   max   
split                                                                         
train             55.0  74.181818  10.811672  25.0  74.50  78.0  79.5  80.0   
validation        12.0  72.083333  11.293267  40.0  70.75  76.5  79.0  80.0   

phase      online                                                           
            count        mean        std   min    25%    50%    75%    max  
split                                                                       
train        55.0  144.527273  24.671812  49.0  147.5  154.0  156.5  160.0  
validation   12.0  139.166667  27.385575  82.0  139.0  152.5  155.0  159.0

Test participants loaded: 0


## 3. Continuous physiological cleaning and baseline helpers

ICA uses the convergent deflation configuration established in the physiological-cleaning notebook. At most three components correlated at 0.30 or higher with EOG are removed, followed by residual EOG regression.

Baseline covariance uses clean non-overlapping two-second windows. Windows with unusually high EEG, EOG, or EMG log power are rejected using a robust z threshold of four.

In [3]:
def robust_z(values):
    values = np.asarray(values, dtype=float)
    median = np.median(values)
    scale = 1.4826 * np.median(np.abs(values - median))
    if not np.isfinite(scale) or scale <= 1e-12:
        scale = np.std(values)
    if not np.isfinite(scale) or scale <= 1e-12:
        return np.zeros_like(values)
    return (values - median) / scale

def normalized_covariance(epoch):
    epoch = epoch - epoch.mean(axis=1, keepdims=True)
    covariance = epoch @ epoch.T
    trace = np.trace(covariance)
    if not np.isfinite(trace) or trace <= 0:
        raise ValueError('Invalid covariance trace.')
    return covariance / trace

def clean_continuous_recording(path):
    raw = mne.io.read_raw_gdf(path, preload=True, verbose='ERROR')
    try:
        sfreq = float(raw.info['sfreq'])
        eeg = raw.get_data(picks=EEG_CHANNELS)
        eog = raw.get_data(picks=EOG_CHANNELS)
        emg = raw.get_data(picks=EMG_CHANNELS)
    finally:
        raw.close()
    assert sfreq == 512.0

    eeg_car = eeg - eeg.mean(axis=0, keepdims=True)
    eeg_1_40 = signal.sosfiltfilt(
        signal.butter(4, [1.0, 40.0], btype='bandpass', fs=sfreq, output='sos'),
        eeg_car, axis=1,
    )
    eog_1_15 = signal.sosfiltfilt(
        signal.butter(4, [1.0, 15.0], btype='bandpass', fs=sfreq, output='sos'),
        eog, axis=1,
    )
    positions = np.arange(0, eeg_1_40.shape[1], 4)
    if len(positions) > ICA_FIT_SAMPLES:
        positions = positions[
            np.linspace(0, len(positions) - 1, ICA_FIT_SAMPLES, dtype=int)
        ]
    ica = FastICA(
        n_components=ICA_COMPONENTS,
        whiten='unit-variance',
        algorithm='deflation',
        random_state=SEED,
        max_iter=500,
        tol=5e-3,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        ica.fit(eeg_1_40[:, positions].T)
    if any(issubclass(item.category, ConvergenceWarning) for item in caught):
        raise RuntimeError(f'ICA did not converge for {path}.')

    correlation_positions = np.arange(0, eeg_1_40.shape[1], 4)
    sources = ica.transform(eeg_1_40[:, correlation_positions].T)
    eog_decimated = eog_1_15[:, correlation_positions].T
    correlations = np.corrcoef(
        sources.T, eog_decimated.T
    )[:ICA_COMPONENTS, ICA_COMPONENTS:]
    scores = np.max(np.abs(correlations), axis=1)
    eligible = np.flatnonzero(scores >= ICA_CORRELATION_THRESHOLD)
    removed = eligible[np.argsort(scores[eligible])[::-1]][:MAX_ICA_COMPONENTS]
    full_sources = ica.transform(eeg_1_40.T)
    full_sources[:, removed] = 0.0
    eeg_ica = ica.inverse_transform(full_sources).T

    eog_centered = eog - eog.mean(axis=1, keepdims=True)
    eeg_centered = eeg_ica - eeg_ica.mean(axis=1, keepdims=True)
    gram = eog_centered @ eog_centered.T
    ridge = max(np.trace(gram) / 3 * 1e-6, np.finfo(float).eps)
    coefficients = (
        eeg_centered @ eog_centered.T
        @ np.linalg.inv(gram + ridge * np.eye(3))
    )
    cleaned = eeg_centered - coefficients @ eog_centered
    return cleaned, eog, emg, sfreq, len(removed)

def participant_baseline_paths(participant):
    participant_rows = development.loc[
        development['participant'].eq(participant)
    ]
    source = Path(participant_rows['source_file'].iloc[0])
    folder = SIGNALS_ROOT / source.parent
    return (
        folder / f'{participant}_OE_baseline.gdf',
        folder / f'{participant}_CE_baseline.gdf',
    )

def individual_mu_peak(cleaned_ce, sfreq):
    central = [EEG_CHANNELS.index(name) for name in ['C3', 'C4', 'Cz']]
    frequencies, spectrum = signal.welch(
        cleaned_ce[central], fs=sfreq, nperseg=int(4 * sfreq),
        noverlap=int(2 * sfreq), axis=1,
    )
    mean_spectrum = spectrum.mean(axis=0)
    selection = (frequencies >= 7.0) & (frequencies <= 14.0)
    return float(frequencies[selection][np.argmax(mean_spectrum[selection])])

def average_clean_baseline_covariances(cleaned, eog, emg, sfreq, bands):
    band_signals = [
        signal.sosfiltfilt(
            signal.butter(4, limits, btype='bandpass', fs=sfreq, output='sos'),
            cleaned, axis=1,
        )
        for limits in bands.values()
    ]
    eog_filtered = signal.sosfiltfilt(
        signal.butter(4, [1.0, 15.0], btype='bandpass', fs=sfreq, output='sos'),
        eog, axis=1,
    )
    emg_filtered = signal.sosfiltfilt(
        signal.butter(4, [30.0, 100.0], btype='bandpass', fs=sfreq, output='sos'),
        emg, axis=1,
    )
    window = int(round(BASELINE_WINDOW_SECONDS * sfreq))
    starts = np.arange(0, cleaned.shape[1] - window + 1, window)
    covariances = np.empty((len(starts), 2, 27, 27), dtype=float)
    quality = np.empty((len(starts), 3), dtype=float)
    for number, start in enumerate(starts):
        stop = start + window
        for band_number, filtered in enumerate(band_signals):
            covariances[number, band_number] = normalized_covariance(
                filtered[:, start:stop]
            )
        quality[number, 0] = np.log(np.mean(np.square(cleaned[:, start:stop])))
        quality[number, 1] = np.log(np.mean(np.square(eog_filtered[:, start:stop])))
        quality[number, 2] = np.log(np.mean(np.square(emg_filtered[:, start:stop])))
    quality_z = np.column_stack([
        robust_z(quality[:, column]) for column in range(3)
    ])
    keep = np.all(quality_z < 4.0, axis=1)
    if keep.sum() < 10:
        raise ValueError('Fewer than 10 clean eyes-open baseline windows.')
    return covariances[keep].mean(axis=0), int(keep.sum()), int((~keep).sum())

print('Baseline-cleaning helpers ready.')

Baseline-cleaning helpers ready.


## 4. Extract participant baselines

Eyes-closed baseline estimates the individual mu peak because alpha/mu activity is usually clearer with closed eyes. Eyes-open baseline supplies covariance alignment because its behavioral state is closer to the motor-imagery task.

In [4]:
def baseline_cache_path(participant):
    return BASELINE_CACHE_ROOT / f'{participant}_baseline_calibration.npz'

baseline_rows = []
all_participants = sorted(
    participant_sets['train'] | participant_sets['validation']
)
baseline_started = time.perf_counter()

for number, participant in enumerate(all_participants, start=1):
    destination = baseline_cache_path(participant)
    if destination.is_file():
        try:
            with np.load(destination, allow_pickle=False) as saved:
                valid = (
                    str(saved['participant']) == participant
                    and saved['fixed_covariances'].shape == (2, 27, 27)
                    and saved['individual_covariances'].shape == (2, 27, 27)
                )
        except Exception:
            valid = False
    else:
        valid = False

    if not valid:
        oe_path, ce_path = participant_baseline_paths(participant)
        cleaned_ce, _, _, sfreq_ce, ce_removed = clean_continuous_recording(ce_path)
        peak = individual_mu_peak(cleaned_ce, sfreq_ce)
        del cleaned_ce
        individual_bands = {
            'mu': (max(6.0, peak - 2.0), min(15.0, peak + 2.0)),
            'beta': FIXED_BANDS['beta'],
        }
        cleaned_oe, eog_oe, emg_oe, sfreq_oe, oe_removed = (
            clean_continuous_recording(oe_path)
        )
        fixed_covariances, fixed_kept, fixed_rejected = (
            average_clean_baseline_covariances(
                cleaned_oe, eog_oe, emg_oe, sfreq_oe, FIXED_BANDS
            )
        )
        individual_covariances, individual_kept, individual_rejected = (
            average_clean_baseline_covariances(
                cleaned_oe, eog_oe, emg_oe, sfreq_oe, individual_bands
            )
        )
        np.savez_compressed(
            destination,
            participant=np.asarray(participant),
            individual_mu_peak_hz=np.asarray(peak),
            individual_mu_band_hz=np.asarray(individual_bands['mu']),
            fixed_covariances=fixed_covariances,
            individual_covariances=individual_covariances,
            fixed_windows_kept=np.asarray(fixed_kept),
            fixed_windows_rejected=np.asarray(fixed_rejected),
            individual_windows_kept=np.asarray(individual_kept),
            individual_windows_rejected=np.asarray(individual_rejected),
            ce_ica_components_removed=np.asarray(ce_removed),
            oe_ica_components_removed=np.asarray(oe_removed),
        )

    with np.load(destination, allow_pickle=False) as saved:
        baseline_rows.append({
            'participant': participant,
            'individual_mu_peak_hz': float(saved['individual_mu_peak_hz']),
            'mu_low_hz': float(saved['individual_mu_band_hz'][0]),
            'mu_high_hz': float(saved['individual_mu_band_hz'][1]),
            'oe_windows_kept': int(saved['fixed_windows_kept']),
            'oe_windows_rejected': int(saved['fixed_windows_rejected']),
            'ce_ica_components_removed': int(saved['ce_ica_components_removed']),
            'oe_ica_components_removed': int(saved['oe_ica_components_removed']),
        })
    if number == 1 or number % 10 == 0 or number == len(all_participants):
        print(
            f'[{number:>2}/{len(all_participants)}] {participant} '
            f'({(time.perf_counter() - baseline_started) / 60:.1f} min)'
        )

baseline_summary = pd.DataFrame(baseline_rows)
baseline_summary.to_csv(OUTPUT_ROOT / 'baseline_calibration_summary.csv', index=False)
display(baseline_summary.describe())

[ 1/67] A1 (0.0 min)
[10/67] A21 (0.0 min)
[20/67] A33 (0.0 min)
[30/67] A45 (0.0 min)
[40/67] A55 (0.0 min)
[50/67] B64 (0.0 min)
[60/67] B75 (0.0 min)
[67/67] C86 (0.0 min)


,individual_mu_peak_hz,mu_low_hz,mu_high_hz,oe_windows_kept,oe_windows_rejected,ce_ica_components_removed,oe_ica_components_removed
count,67.000000,67.000000,67.000000,67.000000,67.000000,67.000000,67.000000
mean,9.985075,8.037313,11.973881,84.537313,8.567164,1.253731,1.432836
std,1.385723,1.291914,1.357593,9.002236,8.826997,0.471724,0.556610
min,7.000000,6.000000,9.000000,61.000000,0.000000,1.000000,1.000000
25%,9.250000,7.250000,11.250000,79.500000,1.000000,1.000000,1.000000
50%,10.250000,8.250000,12.250000,87.000000,6.000000,1.000000,1.000000
75%,10.750000,8.750000,12.750000,92.000000,13.500000,1.000000,2.000000
max,13.750000,11.750000,15.000000,100.000000,32.000000,3.000000,3.000000


## 5. Load fixed-band covariances and compute individual-mu task covariances

The fixed-band matrices come from the physiological-cleaning notebook. Individual-mu matrices require re-filtering each cleaned recording using that participant's baseline-derived mu band. Previously rejected EMG trials remain excluded.

In [5]:
fixed_covariances = np.empty((len(development), 2, 27, 27), dtype=np.float32)
personal_covariances = np.empty_like(fixed_covariances)
y_all = development['label_id'].to_numpy(dtype=np.int64)
participant_array = development['participant'].to_numpy()

for relative_path, rows in development.groupby(
    'physio_covariance_relative_path', sort=True
):
    positions = rows.index.to_numpy(dtype=int)
    covariance_rows = rows['physio_covariance_row'].to_numpy(dtype=int)
    with np.load(DATA_ROOT / relative_path, allow_pickle=False) as saved:
        fixed_covariances[positions] = saved['covariances'][covariance_rows]

def personal_task_cache_path(source_file):
    relative = Path(source_file).with_suffix('')
    return (
        PERSONAL_TASK_ROOT / relative.parent
        / f'{relative.name}_individual_mu_covariances.npz'
    )

def process_personal_task_file(source_file, rows):
    rows = rows.sort_values('sample_id').reset_index()
    participant = rows['participant'].iloc[0]
    expected_ids = rows['sample_id'].to_numpy(dtype=str)
    destination = personal_task_cache_path(source_file)
    if destination.is_file():
        try:
            with np.load(destination, allow_pickle=False) as saved:
                valid = (
                    np.array_equal(saved['sample_ids'].astype(str), expected_ids)
                    and saved['covariances'].shape == (len(rows), 2, 27, 27)
                )
        except Exception:
            valid = False
    else:
        valid = False
    if not valid:
        with np.load(baseline_cache_path(participant), allow_pickle=False) as saved:
            mu_limits = tuple(saved['individual_mu_band_hz'].astype(float))
        cleaned, _, _, sfreq, _ = clean_continuous_recording(
            SIGNALS_ROOT / source_file
        )
        bands = [mu_limits, FIXED_BANDS['beta']]
        filtered = [
            signal.sosfiltfilt(
                signal.butter(4, limits, btype='bandpass', fs=sfreq, output='sos'),
                cleaned, axis=1,
            )
            for limits in bands
        ]
        start_offset = int(round(STIMULUS_SECONDS[0] * sfreq))
        stop_offset = int(round(STIMULUS_SECONDS[1] * sfreq))
        covariances = np.empty((len(rows), 2, 27, 27), dtype=np.float32)
        for row_number, row in enumerate(rows.itertuples(index=False)):
            start = int(row.cue_sample) + start_offset
            stop = int(row.cue_sample) + stop_offset
            for band_number, band_signal in enumerate(filtered):
                covariances[row_number, band_number] = normalized_covariance(
                    band_signal[:, start:stop]
                )
        destination.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            destination,
            sample_ids=expected_ids,
            covariances=covariances,
            individual_mu_band_hz=np.asarray(mu_limits),
            participant=np.asarray(participant),
        )
    return destination, rows['index'].to_numpy(dtype=int)

file_groups = list(development.groupby('source_file', sort=True))
task_started = time.perf_counter()
for number, (source_file, rows) in enumerate(file_groups, start=1):
    cache_path, positions = process_personal_task_file(source_file, rows)
    with np.load(cache_path, allow_pickle=False) as saved:
        personal_covariances[positions] = saved['covariances']
    if number == 1 or number % 25 == 0 or number == len(file_groups):
        print(
            f'[{number:>3}/{len(file_groups)}] individual-mu task covariance '
            f'({(time.perf_counter() - task_started) / 60:.1f} min)'
        )

assert np.isfinite(fixed_covariances).all()
assert np.isfinite(personal_covariances).all()
print('Fixed covariance shape:', fixed_covariances.shape)
print('Individual-mu covariance shape:', personal_covariances.shape)

[  1/389] individual-mu task covariance (0.0 min)
[ 25/389] individual-mu task covariance (0.0 min)
[ 50/389] individual-mu task covariance (0.0 min)
[ 75/389] individual-mu task covariance (0.0 min)
[100/389] individual-mu task covariance (0.0 min)


[125/389] individual-mu task covariance (0.0 min)
[150/389] individual-mu task covariance (0.0 min)
[175/389] individual-mu task covariance (0.0 min)
[200/389] individual-mu task covariance (0.0 min)
[225/389] individual-mu task covariance (0.0 min)


[250/389] individual-mu task covariance (0.0 min)
[275/389] individual-mu task covariance (0.0 min)
[300/389] individual-mu task covariance (0.0 min)
[325/389] individual-mu task covariance (0.0 min)
[350/389] individual-mu task covariance (0.0 min)


[375/389] individual-mu task covariance (0.0 min)
[389/389] individual-mu task covariance (0.0 min)
Fixed covariance shape: (14564, 2, 27, 27)
Individual-mu covariance shape: (14564, 2, 27, 27)


## 6. Alignment, CSP, participant centering, and MLP helpers

Baseline alignment whitens each trial covariance with that participant's eyes-open covariance. CSP and global feature scaling are fitted using only the current training participants.

The acquisition-centering variant subtracts each participant's acquisition-feature mean from all of their trials. This uses acquisition data but not its labels.

In [6]:
baseline_fixed = {}
baseline_personal = {}
for participant in all_participants:
    with np.load(baseline_cache_path(participant), allow_pickle=False) as saved:
        baseline_fixed[participant] = saved['fixed_covariances']
        baseline_personal[participant] = saved['individual_covariances']

def inverse_square_root(covariance):
    covariance = (
        (1.0 - SHRINKAGE) * covariance
        + SHRINKAGE * np.eye(len(covariance)) * np.trace(covariance) / len(covariance)
    )
    values, vectors = linalg.eigh(covariance)
    values = np.maximum(values, 1e-10)
    return (vectors * (1.0 / np.sqrt(values))) @ vectors.T

def align_covariances(covariances, baseline_lookup):
    aligned = np.empty_like(covariances)
    for participant in sorted(set(participant_array)):
        rows = participant_array == participant
        for band in range(2):
            whitening = inverse_square_root(baseline_lookup[participant][band])
            transformed = np.einsum(
                'ab,nbc,cd->nad',
                whitening, covariances[rows, band], whitening,
                optimize=True,
            )
            trace = np.trace(transformed, axis1=1, axis2=2)
            aligned[rows, band] = transformed / trace[:, None, None]
    return aligned

aligned_fixed = align_covariances(fixed_covariances, baseline_fixed)
aligned_personal = align_covariances(personal_covariances, baseline_personal)

def fit_csp(covariances, labels):
    class_covariances = []
    for label in [0, 1]:
        covariance = covariances[labels == label].mean(axis=0)
        target = np.eye(27) * np.trace(covariance) / 27
        class_covariances.append(
            (1.0 - SHRINKAGE) * covariance + SHRINKAGE * target
        )
    values, vectors = linalg.eigh(
        class_covariances[0], class_covariances[0] + class_covariances[1]
    )
    selected = np.r_[
        np.arange(N_CSP_FILTERS_PER_CLASS),
        np.arange(27 - N_CSP_FILTERS_PER_CLASS, 27),
    ]
    return vectors[:, selected].T

def transform_csp(covariances, filters):
    variances = np.einsum(
        'kc,ncd,kd->nk', filters, covariances, filters, optimize=True
    )
    variances = np.maximum(variances, np.finfo(float).eps)
    return np.log(variances / variances.sum(axis=1, keepdims=True))

def build_features(covariances, fit_mask, acquisition_centering=False):
    blocks = []
    for band in range(2):
        filters = fit_csp(covariances[fit_mask, band], y_all[fit_mask])
        blocks.append(transform_csp(covariances[:, band], filters))
    raw = np.column_stack(blocks).astype(np.float32)
    mean = raw[fit_mask].mean(axis=0)
    std = raw[fit_mask].std(axis=0)
    X = ((raw - mean) / std).astype(np.float32)
    if acquisition_centering:
        for participant in sorted(set(participant_array)):
            participant_rows = participant_array == participant
            calibration_rows = participant_rows & development['phase'].eq(
                'acquisition'
            ).to_numpy()
            X[participant_rows] -= X[calibration_rows].mean(axis=0)
    return X

class CalibrationMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(12, 16), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(16, 8), nn.ReLU(), nn.Dropout(0.10),
            nn.Linear(8, 2),
        )

    def forward(self, inputs):
        return self.network(inputs)

def to_tensor(values, dtype):
    values = np.ascontiguousarray(values)
    return torch.frombuffer(memoryview(values), dtype=dtype).reshape(values.shape)

def train_general_model(X, fit_mask, seed, epochs=GENERAL_EPOCHS):
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(
            to_tensor(X[fit_mask], torch.float32),
            to_tensor(y_all[fit_mask], torch.int64),
        ),
        batch_size=BATCH_SIZE, shuffle=True, generator=generator,
    )
    torch.manual_seed(seed)
    model = CalibrationMLP()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    loss_function = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for batch_X, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
    return model

@torch.no_grad()
def predict_probabilities(model, X, mask):
    loader = DataLoader(
        TensorDataset(
            to_tensor(X[mask], torch.float32),
            to_tensor(y_all[mask], torch.int64),
        ),
        batch_size=BATCH_SIZE, shuffle=False,
    )
    model.eval()
    probabilities = []
    for batch_X, _ in loader:
        probabilities.extend(torch.softmax(model(batch_X), dim=1)[:, 1].tolist())
    return np.asarray(probabilities)

def adapt_final_layer(general_model, X, calibration_mask, seed):
    adapted = copy.deepcopy(general_model)
    for parameter in adapted.parameters():
        parameter.requires_grad = False
    for parameter in adapted.network[-1].parameters():
        parameter.requires_grad = True
    loader = DataLoader(
        TensorDataset(
            to_tensor(X[calibration_mask], torch.float32),
            to_tensor(y_all[calibration_mask], torch.int64),
        ),
        batch_size=min(32, int(calibration_mask.sum())), shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    optimizer = torch.optim.AdamW(
        adapted.network[-1].parameters(),
        lr=ADAPTATION_LR, weight_decay=1e-3,
    )
    loss_function = nn.CrossEntropyLoss()
    for _ in range(ADAPTATION_EPOCHS):
        adapted.train()
        for batch_X, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(adapted(batch_X), batch_y)
            loss.backward()
            optimizer.step()
    return adapted

VARIANTS = {
    'uncalibrated_fixed_bands': {
        'covariances': fixed_covariances, 'center': False,
    },
    'baseline_aligned_fixed_bands': {
        'covariances': aligned_fixed, 'center': False,
    },
    'baseline_aligned_individual_mu': {
        'covariances': aligned_personal, 'center': False,
    },
    'baseline_individual_mu_acquisition_centered': {
        'covariances': aligned_personal, 'center': True,
    },
}
print('Calibration variants:', list(VARIANTS))

Calibration variants: ['uncalibrated_fixed_bands', 'baseline_aligned_fixed_bands', 'baseline_aligned_individual_mu', 'baseline_individual_mu_acquisition_centered']


## 7. Select the calibration representation using training participants only

Five participant folds compare the four representations. In every fold, CSP and the MLP are fitted without the held-out participants. Performance is measured only on those participants' online trials.

In [7]:
train_participants = np.asarray(sorted(participant_sets['train']))
rng = np.random.default_rng(SEED)
fold_assignment = {
    participant: fold
    for fold, members in enumerate(
        np.array_split(rng.permutation(train_participants), 5), start=1
    )
    for participant in members
}
train_mask = development['split'].eq('train').to_numpy()
validation_mask = development['split'].eq('validation').to_numpy()
online_mask = development['phase'].eq('online').to_numpy()
acquisition_mask = development['phase'].eq('acquisition').to_numpy()

variant_prediction_frames = []
for fold in range(1, 6):
    held_out = {
        participant for participant, assigned in fold_assignment.items()
        if assigned == fold
    }
    fit_mask = train_mask & ~np.isin(participant_array, list(held_out))
    evaluation_mask = (
        train_mask & online_mask
        & np.isin(participant_array, list(held_out))
    )
    for variant_number, (variant_name, configuration) in enumerate(
        VARIANTS.items(), start=1
    ):
        X = build_features(
            configuration['covariances'], fit_mask,
            acquisition_centering=configuration['center'],
        )
        model = train_general_model(X, fit_mask, SEED + fold * 100)
        probabilities = predict_probabilities(model, X, evaluation_mask)
        rows = development.loc[evaluation_mask, [
            'sample_id', 'participant', 'label_id'
        ]].reset_index(drop=True)
        rows['right_probability'] = probabilities
        rows['prediction'] = (probabilities >= 0.5).astype(int)
        rows['variant'] = variant_name
        rows['fold'] = fold
        variant_prediction_frames.append(rows)
    print(f'Fold {fold}/5 complete.')

variant_predictions = pd.concat(variant_prediction_frames, ignore_index=True)
variant_rows = []
for variant, rows in variant_predictions.groupby('variant'):
    variant_rows.append({
        'variant': variant,
        'participants': rows['participant'].nunique(),
        'online_trials': len(rows),
        'accuracy': accuracy_score(rows['label_id'], rows['prediction']),
        'balanced_accuracy': balanced_accuracy_score(
            rows['label_id'], rows['prediction']
        ),
        'macro_f1': f1_score(
            rows['label_id'], rows['prediction'], average='macro'
        ),
        'roc_auc': roc_auc_score(
            rows['label_id'], rows['right_probability']
        ),
    })
variant_comparison = pd.DataFrame(variant_rows).sort_values(
    'balanced_accuracy', ascending=False
).reset_index(drop=True)
display(variant_comparison)
variant_predictions.to_csv(
    OUTPUT_ROOT / 'training_cv_variant_predictions.csv', index=False
)
variant_comparison.to_csv(
    OUTPUT_ROOT / 'training_cv_variant_comparison.csv', index=False
)
best_variant_name = variant_comparison.iloc[0]['variant']
print('Training-CV winner:', best_variant_name)

Fold 1/5 complete.


Fold 2/5 complete.


Fold 3/5 complete.


Fold 4/5 complete.


Fold 5/5 complete.


,variant,participants,online_trials,accuracy,balanced_accuracy,macro_f1,roc_auc
0,uncalibrated_fixed_bands,55,7949,0.653667,0.653639,0.653621,0.714058
1,baseline_individual_mu_acquisition_centered,55,7949,0.647251,0.647158,0.646841,0.700707
2,baseline_aligned_fixed_bands,55,7949,0.646119,0.646072,0.646003,0.707682
3,baseline_aligned_individual_mu,55,7949,0.637816,0.637781,0.637747,0.688436


Training-CV winner: uncalibrated_fixed_bands


## 8. Tune labeled adaptation and blending on training participants

For the winning representation, each held-out training participant calibrates only the final MLP layer using acquisition trials. General and adapted probabilities are blended. The blend weight is chosen from training-participant online predictions only.

In [8]:
best_configuration = VARIANTS[best_variant_name]
adaptation_frames = []

for fold in range(1, 6):
    held_out = {
        participant for participant, assigned in fold_assignment.items()
        if assigned == fold
    }
    fit_mask = train_mask & ~np.isin(participant_array, list(held_out))
    X = build_features(
        best_configuration['covariances'], fit_mask,
        acquisition_centering=best_configuration['center'],
    )
    general_model = train_general_model(X, fit_mask, SEED + fold * 100)

    for participant_number, participant in enumerate(sorted(held_out), start=1):
        calibration_mask = (
            train_mask & acquisition_mask & (participant_array == participant)
        )
        evaluation_mask = (
            train_mask & online_mask & (participant_array == participant)
        )
        adapted_model = adapt_final_layer(
            general_model, X, calibration_mask,
            SEED + fold * 1000 + participant_number,
        )
        general_probability = predict_probabilities(
            general_model, X, evaluation_mask
        )
        adapted_probability = predict_probabilities(
            adapted_model, X, evaluation_mask
        )
        rows = development.loc[evaluation_mask, [
            'sample_id', 'participant', 'label_id'
        ]].reset_index(drop=True)
        rows['general_probability'] = general_probability
        rows['adapted_probability'] = adapted_probability
        rows['fold'] = fold
        adaptation_frames.append(rows)
    print(f'Adaptation fold {fold}/5 complete.')

training_adaptation_predictions = pd.concat(
    adaptation_frames, ignore_index=True
)
alpha_rows = []
for alpha in np.linspace(0.0, 1.0, 21):
    probability = (
        (1.0 - alpha) * training_adaptation_predictions['general_probability']
        + alpha * training_adaptation_predictions['adapted_probability']
    )
    prediction = (probability >= 0.5).astype(int)
    alpha_rows.append({
        'adapted_weight': alpha,
        'balanced_accuracy': balanced_accuracy_score(
            training_adaptation_predictions['label_id'], prediction
        ),
        'accuracy': accuracy_score(
            training_adaptation_predictions['label_id'], prediction
        ),
    })
alpha_comparison = pd.DataFrame(alpha_rows).sort_values(
    ['balanced_accuracy', 'adapted_weight'], ascending=[False, True]
)
best_alpha = float(alpha_comparison.iloc[0]['adapted_weight'])
display(alpha_comparison)
print('Selected adapted-model weight:', best_alpha)

training_adaptation_predictions.to_csv(
    OUTPUT_ROOT / 'training_cv_adaptation_predictions.csv', index=False
)
alpha_comparison.to_csv(
    OUTPUT_ROOT / 'training_cv_blend_weights.csv', index=False
)

Adaptation fold 1/5 complete.


Adaptation fold 2/5 complete.


Adaptation fold 3/5 complete.


Adaptation fold 4/5 complete.


Adaptation fold 5/5 complete.


,adapted_weight,balanced_accuracy,accuracy
19,0.95,0.654522,0.654548
20,1.00,0.654522,0.654548
18,0.90,0.654395,0.654422
14,0.70,0.654395,0.654422
8,0.40,0.654394,0.654422
4,0.20,0.654393,0.654422
5,0.25,0.654393,0.654422
6,0.30,0.654393,0.654422
9,0.45,0.654268,0.654296
7,0.35,0.654268,0.654296


Selected adapted-model weight: 0.9500000000000001


## 9. Final validation-participant simulation

Each representation is trained on all 55 training participants and evaluated on the 12 validation participants' online trials. For the training-selected representation, the final layer is additionally adapted per participant using only their acquisition trials.

In [9]:
validation_variant_frames = []
trained_models = {}
trained_features = {}

for variant_number, (variant_name, configuration) in enumerate(
    VARIANTS.items(), start=1
):
    X = build_features(
        configuration['covariances'], train_mask,
        acquisition_centering=configuration['center'],
    )
    general_model = train_general_model(X, train_mask, SEED)
    evaluation_mask = validation_mask & online_mask
    probabilities = predict_probabilities(general_model, X, evaluation_mask)
    rows = development.loc[evaluation_mask, [
        'sample_id', 'participant', 'label_id'
    ]].reset_index(drop=True)
    rows['right_probability'] = probabilities
    rows['prediction'] = (probabilities >= 0.5).astype(int)
    rows['variant'] = variant_name
    validation_variant_frames.append(rows)
    trained_models[variant_name] = general_model
    trained_features[variant_name] = X

validation_variant_predictions = pd.concat(
    validation_variant_frames, ignore_index=True
)
validation_variant_rows = []
for variant, rows in validation_variant_predictions.groupby('variant'):
    validation_variant_rows.append({
        'variant': variant,
        'participants': rows['participant'].nunique(),
        'online_trials': len(rows),
        'accuracy': accuracy_score(rows['label_id'], rows['prediction']),
        'balanced_accuracy': balanced_accuracy_score(
            rows['label_id'], rows['prediction']
        ),
        'macro_f1': f1_score(
            rows['label_id'], rows['prediction'], average='macro'
        ),
        'roc_auc': roc_auc_score(
            rows['label_id'], rows['right_probability']
        ),
    })
validation_variant_comparison = pd.DataFrame(
    validation_variant_rows
).sort_values('balanced_accuracy', ascending=False)
display(validation_variant_comparison)

best_general_model = trained_models[best_variant_name]
best_X = trained_features[best_variant_name]
validation_adaptation_frames = []

for participant_number, participant in enumerate(
    sorted(participant_sets['validation']), start=1
):
    calibration_mask = (
        validation_mask & acquisition_mask
        & (participant_array == participant)
    )
    evaluation_mask = (
        validation_mask & online_mask
        & (participant_array == participant)
    )
    adapted_model = adapt_final_layer(
        best_general_model, best_X, calibration_mask,
        SEED + 10_000 + participant_number,
    )
    general_probability = predict_probabilities(
        best_general_model, best_X, evaluation_mask
    )
    adapted_probability = predict_probabilities(
        adapted_model, best_X, evaluation_mask
    )
    blended_probability = (
        (1.0 - best_alpha) * general_probability
        + best_alpha * adapted_probability
    )
    rows = development.loc[evaluation_mask, [
        'sample_id', 'participant', 'label_id'
    ]].reset_index(drop=True)
    rows['general_probability'] = general_probability
    rows['adapted_probability'] = adapted_probability
    rows['blended_probability'] = blended_probability
    validation_adaptation_frames.append(rows)

validation_adaptation = pd.concat(
    validation_adaptation_frames, ignore_index=True
)
method_probabilities = {
    'general': validation_adaptation['general_probability'],
    'adapted_final_layer': validation_adaptation['adapted_probability'],
    'training_tuned_blend': validation_adaptation['blended_probability'],
}
adaptation_metric_rows = []
for method, probabilities in method_probabilities.items():
    predictions = (probabilities >= 0.5).astype(int)
    adaptation_metric_rows.append({
        'method': method,
        'participants': validation_adaptation['participant'].nunique(),
        'online_trials': len(validation_adaptation),
        'accuracy': accuracy_score(
            validation_adaptation['label_id'], predictions
        ),
        'balanced_accuracy': balanced_accuracy_score(
            validation_adaptation['label_id'], predictions
        ),
        'macro_f1': f1_score(
            validation_adaptation['label_id'], predictions, average='macro'
        ),
        'roc_auc': roc_auc_score(
            validation_adaptation['label_id'], probabilities
        ),
    })
adaptation_metrics = pd.DataFrame(adaptation_metric_rows)
display(adaptation_metrics)

participant_rows = []
for participant, rows in validation_adaptation.groupby('participant'):
    general_prediction = (rows['general_probability'] >= 0.5).astype(int)
    blended_prediction = (rows['blended_probability'] >= 0.5).astype(int)
    general_balanced = balanced_accuracy_score(
        rows['label_id'], general_prediction
    )
    blended_balanced = balanced_accuracy_score(
        rows['label_id'], blended_prediction
    )
    participant_rows.append({
        'participant': participant,
        'calibration_trials': int(
            (
                validation_mask & acquisition_mask
                & (participant_array == participant)
            ).sum()
        ),
        'evaluation_trials': len(rows),
        'general_balanced_accuracy': general_balanced,
        'calibrated_balanced_accuracy': blended_balanced,
        'change': blended_balanced - general_balanced,
    })
participant_improvements = pd.DataFrame(participant_rows).sort_values(
    'change', ascending=False
)
display(participant_improvements)
display(participant_improvements['change'].describe())
change_values = participant_improvements['change'].to_numpy()
bootstrap_rng = np.random.default_rng(SEED)
bootstrap_means = np.asarray([
    bootstrap_rng.choice(
        change_values, size=len(change_values), replace=True
    ).mean()
    for _ in range(10_000)
])
change_summary = pd.DataFrame([{
    'participants': len(change_values),
    'mean_change': change_values.mean(),
    'median_change': np.median(change_values),
    'bootstrap_95_ci_low': np.quantile(bootstrap_means, 0.025),
    'bootstrap_95_ci_high': np.quantile(bootstrap_means, 0.975),
    'participants_improved': int((change_values > 1e-12).sum()),
    'participants_unchanged': int((np.abs(change_values) <= 1e-12).sum()),
    'participants_worsened': int((change_values < -1e-12).sum()),
}])
display(change_summary)

validation_variant_predictions.to_csv(
    OUTPUT_ROOT / 'validation_variant_predictions.csv', index=False
)
validation_variant_comparison.to_csv(
    OUTPUT_ROOT / 'validation_variant_comparison.csv', index=False
)
validation_adaptation.to_csv(
    OUTPUT_ROOT / 'validation_adaptation_predictions.csv', index=False
)
adaptation_metrics.to_csv(
    OUTPUT_ROOT / 'validation_adaptation_metrics.csv', index=False
)
participant_improvements.to_csv(
    OUTPUT_ROOT / 'validation_participant_improvements.csv', index=False
)
change_summary.to_csv(
    OUTPUT_ROOT / 'validation_calibration_change_summary.csv', index=False
)

torch.save({
    'model_state_dict': best_general_model.state_dict(),
    'variant': best_variant_name,
    'blend_adapted_weight': best_alpha,
    'general_epochs': GENERAL_EPOCHS,
    'adaptation_epochs': ADAPTATION_EPOCHS,
    'trained': True,
    'tested': False,
}, OUTPUT_ROOT / 'participant_calibrated_general_model.pt')

print('Training-selected variant:', best_variant_name)
print('Training-selected adapted weight:', best_alpha)
print('Test split evaluated: NO')

,variant,participants,online_trials,accuracy,balanced_accuracy,macro_f1,roc_auc
0,baseline_aligned_fixed_bands,12,1670,0.632335,0.631810,0.624044,0.684117
3,uncalibrated_fixed_bands,12,1670,0.619162,0.618653,0.611130,0.696935
1,baseline_aligned_individual_mu,12,1670,0.610180,0.609544,0.597270,0.645958
2,baseline_individual_mu_acquisition_centered,12,1670,0.574251,0.574179,0.574042,0.630783


,method,participants,online_trials,accuracy,balanced_accuracy,macro_f1,roc_auc
0,general,12,1670,0.619162,0.618653,0.611130,0.696935
1,adapted_final_layer,12,1670,0.619760,0.619246,0.611536,0.697620
2,training_tuned_blend,12,1670,0.619760,0.619246,0.611536,0.697594


,participant,calibration_trials,evaluation_trials,general_balanced_accuracy,calibrated_balanced_accuracy,change
5,A45,79,155,0.443973,0.450716,0.006743
4,A31,78,154,0.649351,0.655844,0.006494
0,A1,40,140,0.704559,0.704559,0.000000
1,A20,71,136,0.761039,0.761039,0.000000
2,A21,73,82,0.570048,0.570048,0.000000
3,A28,75,155,0.548384,0.548384,0.000000
6,A48,63,83,0.654360,0.654360,0.000000
8,B65,79,151,0.811293,0.811293,0.000000
9,B66,70,154,0.559241,0.559241,0.000000
10,B77,79,157,0.471429,0.471429,0.000000


count    12.000000
mean      0.000582
std       0.003339
min      -0.006250
25%       0.000000
50%       0.000000
75%       0.000000
max       0.006743
Name: change, dtype: float64

,participants,mean_change,median_change,bootstrap_95_ci_low,bootstrap_95_ci_high,participants_improved,participants_unchanged,participants_worsened
0,12,0.000582,0.0,-0.001042,0.002248,2,9,1


Training-selected variant: uncalibrated_fixed_bands
Training-selected adapted weight: 0.9500000000000001
Test split evaluated: NO


## Interpretation

The no-calibration, baseline-alignment, individual-frequency, acquisition-centering, final-layer adaptation, and blended results are directly comparable because they evaluate the same later online trials.

The test participants remain untouched. This notebook is a development experiment, not a final test estimate.